In [0]:
%sql
CREATE OR REPLACE TABLE medical_insurance.gold.financial_summary AS
WITH visit_revenue AS (
  SELECT 
    v.hospital_id,
    v.visit_date,
    YEAR(v.visit_date) AS revenue_year,
    MONTH(v.visit_date) AS revenue_month,
    QUARTER(v.visit_date) AS revenue_quarter,
    v.visit_type,
    v.department_id,
    COUNT(DISTINCT v.visit_id) AS visit_count,
    SUM(v.total_amount) AS total_visit_revenue
  FROM medical_insurance.silver.visit_silver v
  GROUP BY v.hospital_id, v.visit_date, v.visit_type, v.department_id
),
procedure_revenue AS (
  SELECT 
    v.hospital_id,
    vp.procedure_date,
    YEAR(vp.procedure_date) AS proc_year,
    MONTH(vp.procedure_date) AS proc_month,
    COUNT(DISTINCT vp.visit_procedure_id) AS procedure_count,
    SUM(vp.procedure_amount) AS total_procedure_revenue
  FROM medical_insurance.silver.visit_procedure_silver vp
  JOIN medical_insurance.silver.visit_silver v ON vp.visit_id = v.visit_id
  GROUP BY v.hospital_id, vp.procedure_date
),
claim_financials AS (
  SELECT 
    c.hospital_id,
    c.claim_date,
    YEAR(c.claim_date) AS claim_year,
    MONTH(c.claim_date) AS claim_month,
    COUNT(DISTINCT c.claim_id) AS claim_count,
    SUM(c.claim_amount) AS total_claimed,
    SUM(c.approved_amount) AS total_approved,
    SUM(c.claim_amount - c.approved_amount) AS total_denied
  FROM medical_insurance.silver.claim_silver c
  GROUP BY c.hospital_id, c.claim_date
),
hospital_info AS (
  SELECT 
    h.hospital_id,
    h.hospital_name,
    h.hospital_type,
    h.governorate
  FROM medical_insurance.silver.hospital_silver h
),
department_info AS (
  SELECT 
    dept.department_id,
    dept.hospital_id,
    dept.department_name
  FROM medical_insurance.silver.department_silver dept
),
aggregated_financials AS (
  SELECT 
    vr.hospital_id,
    vr.visit_date AS transaction_date,
    vr.revenue_year,
    vr.revenue_month,
    vr.revenue_quarter,
    vr.visit_type,
    vr.department_id,
    vr.visit_count,
    vr.total_visit_revenue,
    pr.procedure_count,
    pr.total_procedure_revenue,
    cf.claim_count,
    cf.total_claimed,
    cf.total_approved,
    cf.total_denied
  FROM visit_revenue vr
  LEFT JOIN procedure_revenue pr 
    ON vr.hospital_id = pr.hospital_id 
    AND vr.visit_date = pr.procedure_date
  LEFT JOIN claim_financials cf 
    ON vr.hospital_id = cf.hospital_id 
    AND vr.visit_date = cf.claim_date
)
SELECT 
  af.hospital_id,
  h.hospital_name,
  h.hospital_type,
  h.governorate,
  af.transaction_date,
  af.revenue_year,
  af.revenue_month,
  af.revenue_quarter,
  af.visit_type,
  af.department_id,
  d.department_name,
  
  -- Visit metrics
  COALESCE(af.visit_count, 0) AS visit_count,
  ROUND(COALESCE(af.total_visit_revenue, 0), 2) AS total_visit_revenue,
  
  -- Procedure metrics
  COALESCE(af.procedure_count, 0) AS procedure_count,
  ROUND(COALESCE(af.total_procedure_revenue, 0), 2) AS total_procedure_revenue,
  
  -- Claim metrics
  COALESCE(af.claim_count, 0) AS claim_count,
  ROUND(COALESCE(af.total_claimed, 0), 2) AS total_claimed_amount,
  ROUND(COALESCE(af.total_approved, 0), 2) AS total_approved_amount,
  ROUND(COALESCE(af.total_denied, 0), 2) AS total_denied_amount,
  
  -- Combined revenue
  ROUND(COALESCE(af.total_visit_revenue, 0) + COALESCE(af.total_procedure_revenue, 0), 2) AS total_revenue,
  
  -- Reimbursement metrics
  CASE 
    WHEN af.total_claimed > 0 THEN ROUND((af.total_approved * 100.0 / af.total_claimed), 2)
    ELSE 0
  END AS reimbursement_rate_pct,
  
  ROUND(COALESCE(af.total_visit_revenue, 0) + COALESCE(af.total_procedure_revenue, 0) - COALESCE(af.total_approved, 0), 2) AS net_revenue_after_claims,
  
  -- Average metrics
  CASE 
    WHEN af.visit_count > 0 THEN ROUND(af.total_visit_revenue / af.visit_count, 2)
    ELSE 0
  END AS avg_revenue_per_visit,
  
  CASE 
    WHEN af.procedure_count > 0 THEN ROUND(af.total_procedure_revenue / af.procedure_count, 2)
    ELSE 0
  END AS avg_revenue_per_procedure,
  
  CASE 
    WHEN af.claim_count > 0 THEN ROUND(af.total_approved / af.claim_count, 2)
    ELSE 0
  END AS avg_approved_per_claim,
  
  CURRENT_TIMESTAMP() AS created_at
  
FROM aggregated_financials af
LEFT JOIN hospital_info h ON af.hospital_id = h.hospital_id
LEFT JOIN department_info d ON af.department_id = d.department_id AND af.hospital_id = d.hospital_id

In [0]:
%sql
-- Display sample records from financial summary gold table
SELECT 
  hospital_name,
  transaction_date,
  visit_type,
  department_name,
  visit_count,
  total_revenue,
  reimbursement_rate_pct,
  net_revenue_after_claims
FROM medical_insurance.gold.financial_summary
ORDER BY transaction_date DESC, total_revenue DESC
LIMIT 10

In [0]:
%sql
-- Summary statistics by hospital, year, and quarter
SELECT 
  hospital_name,
  hospital_type,
  revenue_year,
  revenue_quarter,
  SUM(visit_count) AS total_visits,
  ROUND(SUM(total_revenue), 2) AS total_revenue,
  ROUND(SUM(total_claimed_amount), 2) AS total_claimed,
  ROUND(SUM(total_approved_amount), 2) AS total_approved,
  ROUND(AVG(reimbursement_rate_pct), 2) AS avg_reimbursement_rate,
  ROUND(SUM(net_revenue_after_claims), 2) AS net_revenue
FROM medical_insurance.gold.financial_summary
GROUP BY hospital_name, hospital_type, revenue_year, revenue_quarter
ORDER BY revenue_year DESC, revenue_quarter DESC, total_revenue DESC